# 1. When features outnumber samples (p >> n)

**Learning goals**
* See exactly how ordinary least squares breaks when p > n.
* Understand the minimum-norm solution and why perfect training fit means nothing.
* Meet this project's problem: about 900 cell lines, 18,000 genes.

In genomics, p >> n is the normal case: a few hundred samples, tens of thousands of genes.
With p > n the matrix $X^\top X$ ($p \times p$) has rank at most $n$, so it cannot be inverted.
The equations $X\beta = y$ have infinitely many exact solutions: any $\beta$ plus any vector in
the null space of $X$ (dimension at least $p - n$) fits the training data perfectly.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(7)

## Infinitely many perfect fits

In [2]:
n, p = 30, 100
X = rng.normal(size=(n, p))
beta_true = np.zeros(p); beta_true[:3] = [2, -1.5, 1]
y = X @ beta_true + rng.normal(size=n)

b_min = np.linalg.pinv(X) @ y                    # minimum-norm least squares
null = np.linalg.svd(X)[2][n:].T                 # basis of the null space (p - n columns)
b_other = b_min + null @ rng.normal(size=p - n) * 2
for name, b in [("minimum norm", b_min), ("min norm + null-space vector", b_other)]:
    print(f"{name:<30} train RSS = {np.sum((y - X @ b) ** 2):.1e}   ||b|| = {np.linalg.norm(b):.1f}")
Xt = rng.normal(size=(1000, p)); yt = Xt @ beta_true + rng.normal(size=1000)
for name, b in [("minimum norm", b_min), ("min norm + null-space vector", b_other), ("truth", beta_true)]:
    print(f"{name:<30} test RMSE = {np.sqrt(np.mean((yt - Xt @ b) ** 2)):.2f}")

minimum norm                   train RSS = 6.1e-28   ||b|| = 1.8
min norm + null-space vector   train RSS = 2.6e-27   ||b|| = 18.6
minimum norm                   test RMSE = 2.46
min norm + null-space vector   test RMSE = 18.40
truth                          test RMSE = 0.98


Both fit the training data exactly, and both predict new data badly. The data cannot tell them
apart. **A penalty is a way to choose among the perfect fits**: ridge picks small $\lVert\beta\rVert_2$,
the lasso picks small $\lVert\beta\rVert_1$ (which favours sparse solutions). Minimum-norm OLS is
the limit of ridge as $\lambda \to 0$.

## The same thing on the real data

In [3]:
# The project's data: one drug, its screened cell lines, 2,000 most variable genes
from drugresponse.experiment import Store
from drugresponse.models import Design
S = Store()
drug_name = "Nutlin-3a (-)"
d = S.drugs.index[S.drugs.drug_name == drug_name][0]
ok = S.R[d].notna().values
des = Design(2000).fit(S.Ev[ok], S.tissue[ok], S.sanger[ok])
X = des.transform(S.Ev[ok], S.tissue[ok], S.sanger[ok])[:, ~des.is_tissue()]   # genes only
y = S.R[d].values[ok]
genes = S.genes[des.genes_]
print(f"{drug_name}: {X.shape[0]} cell lines x {X.shape[1]} genes")

Nutlin-3a (-): 943 cell lines x 2000 genes


In [4]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
cv = KFold(5, shuffle=True, random_state=0)
rows = []
for lam in [1e-6, 1e0, 1e2, 1e3, 1e4, 1e5]:
    pred = cross_val_predict(Ridge(alpha=lam), X, y, cv=cv)
    fit = Ridge(alpha=lam).fit(X, y)
    rows.append((lam, np.corrcoef(fit.predict(X), y)[0, 1], np.corrcoef(pred, y)[0, 1]))
pd.DataFrame(rows, columns=["lambda", "training r", "cross-validated r"]).round(3)

,lambda,training r,cross-validated r
0,0.0,0.999,0.572
1,1.0,0.999,0.573
2,100.0,0.983,0.638
3,1000.0,0.898,0.668
4,10000.0,0.708,0.578
5,100000.0,0.479,0.433


With $\lambda \approx 0$ the training correlation is 1 (memorisation) and the cross-validated
correlation is much lower. As $\lambda$ grows the two meet. Somewhere in between is the best
out-of-sample accuracy, and choosing that $\lambda$ honestly is the subject of notebook 5.

**Key takeaways**
* With p > n, OLS has infinitely many perfect fits; the training error is always zero.
* Regularisation chooses one of them using a prior preference (small, or sparse, coefficients).
* Never judge a p >> n model by its training fit.